# Bayesian Analysis of Semantic Alignment in Large Language Models

## 1. Introduction & Research Objective

### 1.1 Background

This prototype extends our previous exploratory analysis of human and LLM-generated word associations.

In the original EDA, we compared associations from **SWOW (human participants)** with those generated by **Llama3** and **Mistral**, using 11,298 aligned cue words.

Our analysis revealed differences in response diversity, concentration, and exact lexical overlap with human associations.

However, the previous overlap metric considered only **distinct responses**, without accounting for how frequently each association was generated.

### 1.2 Research Objective

The objective of this prototype is to investigate whether Mistral and Llama3 differ in their underlying probability of generating associations observed in SWOW.

Rather than relying exclusively on observed averages, we use **Bayesian inference** to estimate these probabilities and quantify uncertainty.

We focus on two research questions:

- **RQ1:** Does Mistral generate human-observed associations more frequently than Llama3?
- **RQ2 (Exploratory):** Does the difference in alignment vary across cue words?

### 1.3 Prototype Scope

The prototype focuses on:

1. Computing response-level alignment from the existing datasets.
2. Defining a simple Bayesian probabilistic model.
3. Estimating posterior distributions for alignment probabilities.
4. Comparing the two LLMs using posterior differences.
5. Evaluating the model through basic diagnostics and posterior predictive checks.
6. Exploring cue-word variability as preparation for the full project.

The hierarchical modeling extension and the relationship between response concentration and alignment are reserved for the full study.

**Note:** Alignment refers to exact agreement with human-observed associations in SWOW, not a direct measure of semantic understanding.

## 2. Dataset Description

This prototype reuses the datasets prepared in our previous EDA, consisting of human and LLM-generated word associations.

### 2.1 Data Sources

- **SWOW (Small World of Words):** Human-generated word associations, used as the reference for observed human responses.
- **LWOW Llama3:** Word associations generated by Llama3.
- **LWOW Mistral:** Word associations generated by Mistral.

All three datasets contain cue words and up to three associated responses per observation (`R1`, `R2`, `R3`).

Our previous EDA identified **11,298 common cue words**, allowing us to compare both LLMs against the same human reference.

### 2.2 Previous EDA Findings

Our exploratory analysis investigated response diversity, concentration, and exact lexical overlap with SWOW.

The mean distinct-response overlap was:

- **Llama3:** 21.26%
- **Mistral:** 42.48%

These values describe the proportion of distinct LLM responses also observed in SWOW. They do not account for how frequently individual responses were generated.

### 2.3 Response-Level Alignment

For this prototype, we introduce a frequency-aware alignment measure.

A generated response is considered a **match** if it appears among the observed SWOW responses for the same cue word.

For each LLM and cue word, we will calculate:

- **Matches:** Number of generated responses matching SWOW.
- **Total responses:** Number of valid generated responses.
- **Observed alignment rate:** Matches divided by total responses.

Unlike distinct-response overlap, this measure counts repeated responses, allowing frequently generated associations to contribute proportionally.

The resulting observations will be used to estimate the underlying alignment probabilities through Bayesian inference.

**Limitation:** Exact lexical agreement with SWOW does not necessarily represent semantic correctness or similarity. Responses absent from SWOW may still be semantically meaningful.

## 3. Data Loading & Preparation

This prototype reuses the cleaned and transformed datasets from our previous EDA.

The preprocessing steps, including normalization and conversion into individual cue-response associations, were performed in `EDA_llm_cognition.ipynb`.

The resulting datasets were exported to `data/processed/`, preserving the original response frequencies.

### 3.1 Loading Preprocessed Data

We load the preprocessed SWOW, Llama3, and Mistral associations and inspect their structure before constructing the observations for Bayesian inference.

In [3]:
import pandas as pd
from pathlib import Path

processed_dir = Path("data/processed")

swow_long = pd.read_csv(processed_dir / "swow_long.csv")
llama_long = pd.read_csv(processed_dir / "llama3_long.csv")
mistral_long = pd.read_csv(processed_dir / "mistral_long.csv")

datasets = {
    "SWOW": swow_long,
    "Llama3": llama_long,
    "Mistral": mistral_long
}

for name, df in datasets.items():
    print(f"\n{name}: {len(df):,} associations")
    display(df.head(3))
    print(df.columns.tolist())



SWOW: 3,403,790 associations


,cue,position,response
0,although,R1,nevertheless
1,deal,R1,no
2,music,R1,notes


['cue', 'position', 'response']

Llama3: 3,348,049 associations


,cue,position,response
0,a,R1,absorbent
1,a,R1,absorbent
2,a,R1,absorbent


['cue', 'position', 'response']

Mistral: 3,268,206 associations


,cue,position,response
0,a,R1,airplane
1,a,R1,also
2,a,R1,an


['cue', 'position', 'response']


### 3.2 Cue Alignment & Validation

To ensure a fair comparison, we restrict the analysis to cue words with valid responses in all three datasets.

We reuse the alignment criterion established in the previous EDA, preserving only cues shared by SWOW, Llama3, and Mistral.

This ensures both LLMs are compared against the same human reference words.

In [4]:
aligned_cues = (
    set(swow_long["cue"])
    & set(llama_long["cue"])
    & set(mistral_long["cue"])
)

print(f"Aligned cue words: {len(aligned_cues):,}")

assert len(aligned_cues) == 11298, "Cue alignment differs from the original EDA."


Aligned cue words: 11,298


### 3.3 Filtering Aligned Associations

We restrict each dataset to the 11,298 common cue words identified in the previous step.

This ensures that all subsequent measurements are calculated over the same set of stimuli, while preserving individual responses and their frequencies.

In [5]:
swow_aligned = swow_long[swow_long["cue"].isin(aligned_cues)].copy()
llama_aligned = llama_long[llama_long["cue"].isin(aligned_cues)].copy()
mistral_aligned = mistral_long[mistral_long["cue"].isin(aligned_cues)].copy()

for name, df in {
    "SWOW": swow_aligned,
    "Llama3": llama_aligned,
    "Mistral": mistral_aligned
}.items():
    print(f"{name}: {len(df):,} associations, {df['cue'].nunique():,} cues")

    assert df["cue"].nunique() == len(aligned_cues)


SWOW: 3,127,866 associations, 11,298 cues
Llama3: 3,329,829 associations, 11,298 cues
Mistral: 3,241,259 associations, 11,298 cues


### 3.4 Response-Level Matching

We identify whether each LLM-generated response appears among the human-observed associations in SWOW for the same cue word.

Unlike the distinct-response overlap measure used in the previous EDA, repeated responses are counted individually.

Each LLM response is classified as:

- **Match (1):** The response appears in SWOW for the same cue.
- **Non-match (0):** The response does not appear in SWOW for that cue.

These binary observations will subsequently be aggregated into success counts and total response counts for Bayesian modeling.

In [6]:
swow_pairs = swow_aligned[["cue", "response"]].drop_duplicates()

llama_matches = llama_aligned.merge(
    swow_pairs.assign(match=1),
    on=["cue", "response"],
    how="left"
)

mistral_matches = mistral_aligned.merge(
    swow_pairs.assign(match=1),
    on=["cue", "response"],
    how="left"
)

llama_matches["match"] = llama_matches["match"].fillna(0).astype(int)
mistral_matches["match"] = mistral_matches["match"].fillna(0).astype(int)

display(llama_matches.head())
display(mistral_matches.head())


,cue,position,response,match
0,a,R1,absorbent,0
1,a,R1,absorbent,0
2,a,R1,absorbent,0
3,a,R1,abundance,0
4,a,R1,acorn,0


,cue,position,response,match
0,a,R1,airplane,0
1,a,R1,also,0
2,a,R1,an,1
3,a,R1,an,1
4,a,R1,an,1


In [7]:
for name, original, matched in [
    ("Llama3", llama_aligned, llama_matches),
    ("Mistral", mistral_aligned, mistral_matches)
]:
    assert len(original) == len(matched)
    assert matched["match"].isin([0, 1]).all()

    print(f"{name}:")
    print(f"  Total responses: {len(matched):,}")
    print(f"  Matched responses: {matched['match'].sum():,}")
    print(f"  Observed match rate: {matched['match'].mean():.2%}")


Llama3:
  Total responses: 3,329,829
  Matched responses: 827,351
  Observed match rate: 24.85%
Mistral:
  Total responses: 3,241,259
  Matched responses: 2,377,671
  Observed match rate: 73.36%


### 3.5 Aggregating Observations per Cue Word

We aggregate the binary response matches for each cue word and LLM.

For every cue-model pair, we calculate the number of matching responses, the total number of valid responses, and the observed alignment rate.

We preserve cue-level observations to support future analysis of variation across words, rather than relying exclusively on overall aggregated rates.

In [8]:
llama_counts = llama_matches.groupby("cue")["match"].agg(
    matches="sum",
    total="count"
).reset_index()

mistral_counts = mistral_matches.groupby("cue")["match"].agg(
    matches="sum",
    total="count"
).reset_index()

llama_counts["model"] = "Llama3"
mistral_counts["model"] = "Mistral"

alignment_data = pd.concat(
    [llama_counts, mistral_counts],
    ignore_index=True
)

alignment_data["alignment_rate"] = (
    alignment_data["matches"] / alignment_data["total"]
)

display(alignment_data.head())
print(f"Observations: {len(alignment_data):,}")


,cue,matches,total,model,alignment_rate
0,a,21,300,Llama3,0.070000
1,a few,91,300,Llama3,0.303333
2,a little,74,300,Llama3,0.246667
3,a lot,60,300,Llama3,0.200000
4,aardvark,33,300,Llama3,0.110000


Observations: 22,596


## 4. Preliminary Data Analysis

Before specifying the Bayesian model, we examine the distribution of observed response-level alignment rates across cue words.

Our previous analysis showed a substantial difference in overall alignment between Llama3 and Mistral. However, these aggregate rates do not reveal how consistently the difference appears across individual cues.

This preliminary analysis examines cue-level variation and establishes a descriptive baseline for subsequent posterior inference.

### 4.1 Alignment Distribution by Model

We compare the mean, median, and standard deviation of cue-level alignment rates for each LLM.

Unlike the overall response-weighted alignment rate, these summaries give each cue word equal weight.

In [10]:
alignment_summary = alignment_data.groupby("model")["alignment_rate"].agg(
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
)

display(alignment_summary.round(4))


,mean,median,std,min,max
model,,,,,
Llama3,0.2500,0.2267,0.1372,0.0,0.9614
Mistral,0.7297,0.7633,0.1755,0.0,1.0000


### 4.2 Paired Alignment Differences

Since both LLMs are evaluated on the same aligned cue words, we compare their observed alignment rates directly.

For each cue word, we calculate the difference between Mistral and Llama3.

A positive difference indicates higher observed alignment for Mistral, while a negative difference indicates higher observed alignment for Llama3.

This descriptive comparison helps assess whether the overall difference is widespread across cue words or concentrated in particular cases. It also provides a reference for subsequent Bayesian posterior comparisons.

In [11]:
paired_alignment = alignment_data.pivot(
    index="cue",
    columns="model",
    values="alignment_rate"
)

paired_alignment["difference"] = (
    paired_alignment["Mistral"] - paired_alignment["Llama3"]
)

display(paired_alignment.head())

print(f"Mean difference: {paired_alignment['difference'].mean():.4f}")
print(f"Median difference: {paired_alignment['difference'].median():.4f}")
print(f"Mistral higher: {(paired_alignment['difference'] > 0).mean():.2%}")
print(f"Llama3 higher: {(paired_alignment['difference'] < 0).mean():.2%}")
print(f"Equal: {(paired_alignment['difference'] == 0).mean():.2%}")


model,Llama3,Mistral,difference
cue,,,
a,0.070000,0.409483,0.339483
a few,0.303333,0.539267,0.235934
a little,0.246667,0.512281,0.265614
a lot,0.200000,0.588235,0.388235
aardvark,0.110000,0.507634,0.397634


Mean difference: 0.4798
Median difference: 0.5132
Mistral higher: 97.39%
Llama3 higher: 2.59%
Equal: 0.02%


In [13]:

import plotly.express as px

fig = px.histogram(
    paired_alignment,
    x="difference",
    nbins=40,
    title="Cue-Level Alignment Differences",
    labels={
        "difference": "Observed alignment difference (Mistral - Llama3)"
    }
)

fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    xaxis_title="Observed alignment difference (Mistral - Llama3)",
    yaxis_title="Number of cue words",
    showlegend=False
)

fig.show()


### 4.3 Data Preparation & Preliminary Findings

The preprocessing and preliminary analysis established the following:

**1. Data preparation and validation**

- Reused the cleaned SWOW, Llama3, and Mistral datasets from the previous EDA, without repeating normalization.
- Confirmed **11,298 aligned cue words** across all three datasets.
- Constructed response-level matching indicators while preserving repeated associations.
- Aggregated the observations into **22,596 cue-model records**, containing matching response counts, total response counts, and observed alignment rates.
- Successfully validated the resulting data.

**2. Response-level alignment**

The mean cue-level alignment rates were:

- **Llama3:** 25.00%
- **Mistral:** 72.97%

Unlike the distinct-response overlap calculated in the previous EDA, these measurements account for response frequencies.

**3. Cue-level differences**

- The mean observed alignment difference (Mistral − Llama3) was **0.4798**, or approximately **47.98 percentage points**.
- Mistral exhibited higher observed alignment for **97.39%** of cue words.
- Llama3 exhibited higher observed alignment for **2.59%** of cue words.
- The remaining **0.02%** showed equal alignment.

The distribution of paired differences was predominantly positive, indicating that Mistral's advantage is widespread across cue words rather than being driven exclusively by a small subset.

**4. Interpretation and next steps**

These findings provide strong descriptive evidence of higher exact lexical agreement with SWOW for Mistral. However, they do not establish superior semantic understanding or explain the underlying causes of the difference.

Furthermore, the observed rates and differences are point estimates that do not directly quantify uncertainty about the underlying alignment probabilities.

The next stage introduces Bayesian probabilistic modeling to estimate:

- **Posterior A:** The underlying response-level alignment probability for each LLM.
- **Posterior B:** The difference between the two alignment probabilities and its uncertainty.

The variability across cue words will be investigated further in the full study through hierarchical Bayesian modeling.